<a href="https://colab.research.google.com/github/saipuneet07/csa6102-digital-forensics/blob/main/Linux_SSH_Auth_Log_Suspicious_Login_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
AUTH_LINE_RE = re.compile(
r"""(?P<result>Accepted|Failed) password for (?P<user>\S+) from (?P<ip>[\d.]+)
port (?P<port>\d+)""")

def parse_auth_log(lines):
    """Parse raw auth.log lines into structured dicts."""
    entries = []
    for line in lines:
        match = AUTH_LINE_RE.search(line)
        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })
    return entries

def flag_suspicious_logins(entries, trusted_ips):
    """Flag successful logins from untrusted IPs, especially for root."""
    flagged = []
    for e in entries:
        if e["result"] == "Accepted" and e["ip"] not in trusted_ips:
            severity = "HIGH" if e["user"] == "root" else "MEDIUM"
            flagged.append({**e, "severity": severity})
    return flagged